# Sample 5 — A full transformer block

`sample-4` computed multi-head attention on one fixed example. A real transformer block wraps
that computation with two more ingredients — **residual connections** and **layer
normalization** — and follows it with a **feed-forward network** that gets the same treatment.
This notebook builds one full block as plain PyTorch `nn.Module` classes, generalized to work on
a batch of sequences instead of one hardcoded example.

**One block, in order:**

```
x  ->  MultiHeadAttention  ->  Add & Norm  ->  FeedForward  ->  Add & Norm  ->  output
 \_______________________________/              \____________________________/
        residual connection                          residual connection
```


## 1. Multi-head attention, as a reusable module

Same computation as `sample-4`, but using `nn.Linear` for the four projections (`W_Q`, `W_K`,
`W_V`, `W_O`) so PyTorch tracks their gradients, and operating on a batch dimension: input shape
`(batch, seq_len, d_model)` instead of a single `(seq_len, d_model)` example.

We also accept an optional `mask` — additive, with `-inf` in positions that should get zero
attention weight after softmax. Not used yet in this notebook, but `sample-6` needs it for
**causal masking** (a token must not attend to future tokens), so we build the hook in now.

In [1]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch, seq_len, d_model = x.shape
        # (batch, seq_len, d_model) -> (batch, seq_len, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        return x.view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch, num_heads, seq_len, d_k = x.shape
        # (batch, num_heads, seq_len, d_k) -> (batch, seq_len, num_heads, d_k) -> (batch, seq_len, d_model)
        return x.transpose(1, 2).reshape(batch, seq_len, num_heads * d_k)

    def forward(self, x, mask=None):
        Q, K, V = self.split_heads(self.W_Q(x)), self.split_heads(self.W_K(x)), self.split_heads(self.W_V(x))

        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask  # mask has -inf where attention is disallowed
        weights = torch.softmax(scores, dim=-1)

        out = self.combine_heads(weights @ V)
        return self.W_O(out), weights


In [2]:
torch.manual_seed(0)

batch, seq_len, d_model, num_heads = 2, 5, 16, 4
x = torch.randn(batch, seq_len, d_model)

mha = MultiHeadAttention(d_model, num_heads)
attn_out, attn_weights = mha(x)
print("input  shape:", x.shape)
print("output shape:", attn_out.shape, " (unchanged — attention is shape-preserving)")
print("weights shape:", attn_weights.shape, " (batch, num_heads, seq_len, seq_len)")


input  shape: torch.Size([2, 5, 16])
output shape: torch.Size([2, 5, 16])  (unchanged — attention is shape-preserving)
weights shape: torch.Size([2, 4, 5, 5])  (batch, num_heads, seq_len, seq_len)


## 2. Feed-forward network

After attention mixes information *across* tokens, the feed-forward network processes each
token's vector *independently* (the same two-layer MLP is applied at every position) — expand to
a wider hidden dimension, apply a nonlinearity, project back down.

In [3]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

ffn = PositionwiseFeedForward(d_model=16, d_ff=64)
ffn_out = ffn(attn_out)
print("feed-forward output shape:", ffn_out.shape, " (same shape in and out, again)")


feed-forward output shape: torch.Size([2, 5, 16])  (same shape in and out, again)


## 3. Residual connections + layer normalization ("Add & Norm")

Two problems show up once you stack many of these blocks: gradients can vanish on the way back
through so many layers, and each layer's output distribution can drift, making training unstable.
Two standard fixes, applied after *each* sub-layer:

- **Residual connection**: add the sub-layer's input back to its output (`x + Sublayer(x)`), so
  gradients have a direct path backward and the sub-layer only needs to learn a *correction* to
  its input rather than reproducing it from scratch.
- **Layer normalization**: rescale each token's vector to zero mean / unit variance (with learned
  scale and shift), keeping activations in a consistent range from layer to layer.

In [4]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(x, mask=mask)
        x = self.norm1(x + attn_out)      # Add & Norm around attention

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)       # Add & Norm around the feed-forward network

        return x, attn_weights


## 4. Run it

One forward pass on a toy batch, checking shapes at every stage. Note the output shape is
*identical* to the input shape — this is what makes it possible to stack many of these blocks:
each one's output is a valid input to the next.

In [5]:
block = TransformerBlock(d_model=16, num_heads=4, d_ff=64)

x = torch.randn(batch, seq_len, d_model)
out, weights = block(x)

print("input  shape:", x.shape)
print("output shape:", out.shape)
print("num parameters:", sum(p.numel() for p in block.parameters()))

# Sanity check: LayerNorm output should have ~zero mean, ~unit variance per token vector
token_vec = out[0, 0]
print(f"\nfirst output token — mean: {token_vec.mean().item():.4f}, std: {token_vec.std(unbiased=False).item():.4f}")


input  shape: torch.Size([2, 5, 16])
output shape: torch.Size([2, 5, 16])
num parameters: 3280

first output token — mean: -0.0000, std: 1.0000


## 5. Preview: causal masking

`sample-6` stacks several of these blocks into a decoder-only model, where token `i` must not be
allowed to look at tokens `i+1, i+2, ...` (it hasn't "seen" them yet at generation time). That's
enforced with an additive mask: `-inf` above the diagonal, `0` on and below it, added to the raw
scores before softmax — after softmax, those positions become exactly `0`.

In [6]:
def causal_mask(seq_len):
    # upper triangle (excluding diagonal) = -inf, rest = 0
    mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
    return mask  # broadcasts over (batch, num_heads, seq_len, seq_len)

mask = causal_mask(seq_len)
_, causal_weights = block(x, mask=mask)

print("Attention weights for head 0, batch 0 (rows must be lower-triangular):")
print(causal_weights[0, 0].detach().round(decimals=2))


Attention weights for head 0, batch 0 (rows must be lower-triangular):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5200, 0.4800, 0.0000, 0.0000, 0.0000],
        [0.3400, 0.3800, 0.2800, 0.0000, 0.0000],
        [0.2500, 0.2400, 0.2900, 0.2200, 0.0000],
        [0.1800, 0.1200, 0.3300, 0.2300, 0.1500]])


Every row's weights are zero above the diagonal — token `i` only ever attends to tokens `0..i`.

**Next:** `sample-6-mini-gpt-train-and-generate` stacks several `TransformerBlock`s (with this
causal mask applied) into a small decoder-only model, trains it on real text, and generates from
it — the full picture, from tokenization through to inference.